In [1]:
!pip install  langchain_community langchain

In [2]:
import re
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://aws.amazon.com/rds/aurora/?nc2=h_prod_db_aa&trk=4f4d614b-c533-483d-b76b-9d919387c02e&sc_channel=ps")
content = loader.load()

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
# Create the splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(content)

print(f"Created {len(chunks)} chunks")
#for i, chunk in enumerate(chunks, 1):
    #print(f"Chunk {i}: {chunk.page_content}")

Created 21 chunks


In [4]:
from langchain_openai import OpenAIEmbeddings 

# Initialize embeddings
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# Generate embedding for a query
vector = embedding.embed_query("What is Aurora?")

print(len(vector))       # length of the embedding vector
print(vector[:10])       # preview first 10 values


1536
[-0.02918471951042551, -0.05055428400550695, -0.021827483734261757, 0.01536319047449962, -0.04881419089662174, -0.01308122626591771, 0.010661886285581705, -0.035717700656064694, 0.02019423844785196, 0.00748316356693834]


In [5]:
from langchain_community.vectorstores import FAISS
db = FAISS.from_documents(content, embedding)

# Save FAISS index only (no pickle docstore)
db.save_local("my_vectorstore", index_name="faiss_index")

# Later, reload index and reattach your own docstore
db = FAISS.load_local(
    "my_vectorstore",
    embedding,
    index_name="faiss_index",
    allow_dangerous_deserialization=False  # stays safe// 
)



ValueError: The de-serialization relies loading a pickle file. Pickle files can be modified to deliver a malicious payload that results in execution of arbitrary code on your machine.You will need to set `allow_dangerous_deserialization` to `True` to enable deserialization. If you do this, make sure that you trust the source of the data. For example, if you are loading a file that you created, and no that no one else has modified the file, then this is safe to do. Do not set this to `True` if you are loading a file from an untrusted source (e.g., some random site on the internet.).

In [ ]:
import re
results  = db.similarity_search("What is Amazon Aurora?")
for doc in results:
    collapsed = re.sub(r'\n+', '\n', doc.page_content)
   # print(collapsed)

results  = db.similarity_search("What is Amazon Aurora?", k=3)
for doc in results:
    collapsed = re.sub(r'\n+', '\n', doc.page_content)
    print(collapsed)